# 직접 만든 MCP Server를 AI Assistant와 Python Client에서 사용하기

앞 실습에서는 이미 만들어진 MCP Server를 연결했다. 이번에는 `MCPServer`와 `@mcp.tool()`로 공개되는 `add` Tool의 계약을 확인하고, 같은 Server를 JetBrains AI Assistant와 Python SDK Client에서 각각 발견하고 호출한다.

Math 필수 실습은 로컬 덧셈만 수행하므로 모델 API key와 외부 API가 필요하지 않다. 마지막 Weather 선택 확장은 OpenWeatherMap API key와 외부 네트워크가 준비된 경우에 진행한다. Server는 Notebook이 자동으로 시작하지 않으며, 사용자가 각 Server 파일을 별도 프로세스로 실행한다.


## 실습 연결 구조

`math_server.py`는 Tool을 제공하는 Server이다. JetBrains AI Assistant Host와 Notebook의 Python SDK Client는 서로 다른 실행 환경이지만 같은 `/mcp` endpoint와 `add` 계약을 사용한다.

```mermaid
flowchart LR
    P["사용자 · Server 프로세스 시작"] --> S["Math MCP Server\n127.0.0.1:8001/mcp"]
    J["JetBrains AI Assistant Host"] <-->|"Streamable HTTP"| S
    N["Notebook Python SDK Client"] <-->|"Streamable HTTP"| S
    S --> T["add(a: float, b: float) -> float"]
```

그림은 왼쪽의 AI Assistant Host와 아래의 Notebook Client가 가운데의 동일한 Server에 연결하는 순서로 읽는다. 두 실행 환경이 직접 덧셈하는 것이 아니라 Server의 `add` Tool을 호출하고 결과를 돌려받는다는 점이 핵심이다.

### 첫 번째 확인 질문

1. Server 프로세스는 누가 시작하는가?
2. AI Assistant와 Notebook Client는 어느 endpoint에 연결하는가?
3. 실제 Tool을 사용했다는 근거는 어디에서 확인하는가?


## MCP Python SDK 설치하기

`mcp[cli]`는 MCP Server와 Client를 구현하는 Python SDK와 필요한 CLI 의존성을 함께 설치한다. 이 실습은 SDK v2 API를 사용하므로 주 버전을 2로 고정한다. 설치 전의 import 상태가 커널에 남아 있거나 새 패키지가 바로 인식되지 않으면 커널을 한 번 재시작하고 다음 셀부터 실행한다.


In [1]:
%pip install -U "mcp[cli]>=2,<3"


  Using cached mcp-2.0.0-py3-none-any.whl.metadata (7.7 kB)
Using cached mcp-2.0.0-py3-none-any.whl (349 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 1.29.0
    Uninstalling mcp-1.29.0:
      Successfully uninstalled mcp-1.29.0
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-mcp-adapters 0.3.2 requires mcp<2.0.0,>=1.24.0, but you have mcp 2.0.0 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\playdata2\miniforge3\envs\llm_env\python.exe -m pip install --upgrade pip


## Python SDK v2 확인하기

이번 설치 셀에서 준비한 `mcp` 패키지를 사용한다. `MCPServer`는 Server를 만드는 high-level class이고 `Client`는 실행 중인 Server에 연결해 Tool을 발견하고 호출하는 객체이다. 아래 셀의 import 아래에 설치된 `mcp` 주 버전과 `MCPServer`, `Client` class 이름을 확인하는 코드를 작성한다. 이 확인은 네트워크나 Server를 사용하지 않는다.


In [3]:
from importlib.metadata import version

from mcp import Client
from mcp.server import MCPServer

MCP_SDK_VERSION = version('mcp')
print('mcp SDK:', MCP_SDK_VERSION)
print('server class:', MCPServer.__name__)
print('client class:', Client.__name__)


mcp SDK: 2.0.0
server class: MCPServer
client class: Client


## Python 함수를 MCP Tool 계약으로 바꾸기

`MCPServer("math-tools")`는 Server 객체와 식별 이름을 만든다. `@mcp.tool()`은 바로 아래 Python 함수를 외부 Client가 발견할 수 있는 Tool로 등록한다.

`a: float`, `b: float`는 입력 schema의 숫자 속성이 된다. `-> float`는 SDK가 만드는 출력 객체에서 `result` 속성의 숫자 type이 된다. docstring의 문장은 모델과 사용자에게 Tool의 목적을 설명하므로, 이름만 반복하지 않고 입력과 결과를 분명히 적는다.

```python
from mcp.server import MCPServer

mcp = MCPServer("math-tools")

@mcp.tool()
def add(a: float, b: float) -> float:
    """두 숫자 a와 b를 받아 합계를 반환한다."""
    return a + b
```

함수 이름은 Tool 이름 `add`가 되고, Client는 Python 함수 본문을 읽지 않아도 공개된 이름·설명·schema만으로 올바른 arguments를 만들 수 있다.


## 실제 Server 파일 확인하기

실행 파일은 `servers/math_server.py`이다. Notebook의 작업 디렉터리는 `09_mcp` 폴더로 맞춘다. 아래 셀의 `SERVER_PATH`로 파일을 읽고 전체 소스를 출력하는 코드를 작성한다. 파일을 import하거나 Server를 시작하지 않는다.


In [5]:
from pathlib import Path

SERVER_PATH = Path('servers/math_server.py')


## Math Server를 별도 프로세스로 실행하기

`if __name__ == "__main__"` 아래의 `mcp.run()`은 파일을 직접 실행할 때만 동작한다. `transport="streamable-http"`는 HTTP POST로 MCP 메시지를 주고받게 하고, `host="127.0.0.1"`는 접속 범위를 현재 컴퓨터로 제한한다. `port=8001`과 `streamable_http_path="/mcp"`가 합쳐져 Client endpoint가 된다.

PyCharm에서 `servers/math_server.py`를 열고 Run 버튼을 누르거나, `09_mcp` Terminal에서 다음 명령을 실행한다.

```bash
python servers/math_server.py
```

Run 창이 종료되지 않고 요청을 기다리면 정상이다. 이 Notebook에는 Server 자동 시작 코드를 넣지 않는다. 그래야 Server 프로세스, AI Assistant Host, Notebook Client의 생명주기를 분리해서 관찰할 수 있다.


## JetBrains AI Assistant에 Math Server 등록하기

PyCharm의 `Settings > Tools > AI Assistant > Model Context Protocol (MCP)`에서 `Add`를 누르고 HTTP Server를 선택한다. 다음 설정의 최상위 key는 `mcpServers`이고, `lessonMath`는 AI Assistant 화면에서 구분할 이름이며, `url`은 앞에서 사용자가 실행한 Server endpoint이다.

```json
{
  "mcpServers": {
    "lessonMath": {
      "url": "http://127.0.0.1:8001/mcp"
    }
  }
}
```

`Server level`을 `Project`로 선택하고 `OK`, `Apply`를 누른 뒤 Status를 확인한다. 등록은 endpoint를 기억하게 할 뿐 Server 프로세스를 대신 시작하지 않는다.

AI Chat에서 다음과 같이 요청한다.

> lessonMath의 add Tool을 사용해 3과 7을 더하고, Tool에 전달한 인자와 결과를 보여 줘.

Tool 실행 기록에서 이름 `add`, arguments `a=3`, `b=7`, 결과 `10.0`을 확인한다. 자연어 답변만 보고 성공으로 판단하지 않고 Tool 호출 기록까지 확인한다.

### 두 번째 확인 질문

1. Server 프로세스는 누가 시작했는가?
2. AI Assistant는 어느 endpoint에 연결했는가?
3. 실제 Tool을 사용했다는 근거는 무엇인가?


## Python SDK Client의 연결 함수 만들기

이제 AI Assistant와 별개로 Notebook에서 Python SDK Client를 직접 실행한다. URL로 만든 `Client`를 `async with`에 넣으면 연결이 열리고 블록을 나갈 때 닫힌다. 아래 셀의 URL 아래에 Tool 목록을 반환하는 발견 함수와 `add` Tool을 호출하는 비동기 함수를 각각 작성한다. 두 함수는 각각 새 연결을 사용하여 Tool 발견과 호출의 경계를 따로 확인한다.


In [6]:
MATH_MCP_URL = 'http://127.0.0.1:8001/mcp'

async def discover_math_tools():
    # URL은 이미 실행 중인 Streamable HTTP Server의 단일 MCP endpoint이다.
    async with Client(MATH_MCP_URL) as client:
        # ListToolsResult에는 Server가 공개한 Tool 객체 목록이 들어 있다.
        return await client.list_tools()


async def call_add(a: float, b: float):
    async with Client(MATH_MCP_URL) as client:
        # 첫 인자는 발견된 Tool 이름이고 둘째 인자는 schema에 맞춘 arguments dict이다.
        return await client.call_tool(
            'add',
            {'a': a, 'b': b},
        )


## Tool 이름과 schema 발견하기

`list_tools()`는 함수를 실행하지 않고 Server가 공개한 Tool 계약을 가져온다. `input_schema`에서는 `a`와 `b`가 필수 숫자인지 확인한다. `output_schema`에서는 최상위 객체의 `result` 속성이 숫자인지 확인한다.


In [12]:
tools_result = await discover_math_tools()

# 이 Server에는 add 하나만 있으므로 첫 Tool 객체의 계약을 항목별로 확인한다.
add_tool = tools_result.tools[0]
print('name:', add_tool.name)
print('description:', add_tool.description)
print('input schema:', add_tool.input_schema)
print('output schema:', add_tool.output_schema)

# 이름은 call_tool()의 첫 인자와 일치해야 하고 required는 빠뜨릴 수 없는 arguments key이다.
assert add_tool.name == 'add'
assert add_tool.input_schema['required'] == ['a', 'b']
# SDK는 반환값을 result 속성에 넣으므로 최상위 object와 내부 number type을 함께 검증한다.
assert add_tool.output_schema['type'] == 'object'
assert add_tool.output_schema['properties']['result']['type'] == 'number'

name: add
description: 두 숫자 a와 b를 받아 두 수를 더한 후 곱하기 10 해서 반환한다.
input schema: {'type': 'object', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments'}
output schema: {'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}


## add Tool을 호출하고 구조화된 결과 확인하기

`call_tool()`은 `add`와 arguments를 실제 Server로 전송한다. 반환되는 `CallToolResult`의 `is_error`는 MCP Tool 실행 성공 여부이고, `structured_content`는 `-> float` 반환값을 SDK가 `result` key에 넣어 구조화한 결과이다.


In [14]:
add_result = await call_add(3, 7)
print('is_error:', add_result.is_error)
print('structured content:', add_result.structured_content)

assert add_result.is_error is False
assert add_result.structured_content == {'result': 100.0}

is_error: False
structured content: {'result': 100.0}


## Server 재시작과 Client 재연결 확인하기

AI Assistant의 등록 설정과 실행 중인 Server 프로세스는 생명주기가 다르다. PyCharm Run 창에서 Math Server를 Stop하면 설정과 URL은 남지만 새 요청은 연결할 수 없다.

Server를 다시 실행한 뒤 AI Assistant에서는 `Reconnect`를 누르고, Notebook에서는 아래 셀을 다시 실행해 새 `Client` 연결을 연다. 실패를 의도적으로 만드는 코드는 두지 않고, 중지 상태의 연결 실패는 AI Assistant Status에서 관찰한다.


## 선택 확장: 외부 API를 MCP Tool로 감싸기

Math Server는 로컬 Python 함수만 실행했다. Weather Server는 같은 MCP 계약을 사용하지만, Tool 내부에서 OpenWeatherMap API를 호출해 최신 데이터를 가져온다. 이 차이를 통해 MCP Server가 외부 서비스의 인증 정보와 공급자별 HTTP 구현을 Client 뒤로 분리하는 구조를 확인한다.

```mermaid
flowchart LR
    C["AI Assistant 또는 Notebook Client"] <-->|"MCP · Streamable HTTP"| W["Weather MCP Server\n127.0.0.1:8000/mcp"]
    W -->|"HTTPS 요청 · Server의 API key"| O["OpenWeatherMap API"]
    O -->|"날씨 JSON"| W
```

그림은 Client가 도시 이름만 MCP arguments로 보내고, Weather Server가 자신의 환경 변수에 있는 API key로 외부 API를 호출하는 순서로 읽는다. API key는 Notebook, AI Chat, MCP 등록 JSON에 넣지 않는다. 이 확장은 key와 외부 네트워크가 준비된 경우에만 실행한다.


## Weather Server 의존성 준비하기

`requests`는 Server가 OpenWeatherMap에 HTTPS 요청을 보내는 라이브러리이고, `python-dotenv`는 로컬 환경 파일의 값을 프로세스 환경으로 불러오는 도구이다. Math 필수 경로와 분리해 Weather 확장을 진행할 때만 아래 패키지를 확인한다.


In [ ]:
%pip install -U requests python-dotenv


## Weather Server 코드에서 데이터 경계 확인하기

실행 파일은 `servers/weather_server.py`이다. `city` 입력은 OpenWeatherMap의 `q` parameter로 변환되고, 공급자 JSON에서 도시·섭씨 온도·습도·설명을 골라 MCP의 구조화된 결과로 반환한다. 요청에는 10초 timeout과 redirect 차단을 적용하며, 외부 오류 원문에 API key가 포함될 수 있으므로 Client에는 일반화한 오류만 전달한다. 아래 셀의 `WEATHER_SERVER_PATH`로 파일을 읽고 전체 소스를 출력하는 코드를 작성하며 Server는 시작하지 않는다.


In [ ]:
WEATHER_SERVER_PATH = Path('servers/weather_server.py')


## Weather Server 실행과 AI Assistant 등록하기

[OpenWeather 공식 FAQ](https://openweathermap.org/faq)에 따라 계정과 이메일 인증을 마치고 API key를 준비한다. 새 key의 활성화에는 시간이 걸릴 수 있으므로 수업 전에 발급한다. `weather_server.py`는 `find_dotenv(usecwd=True)`로 `09_mcp/.env`를 찾아 `OPENWEATHER_API_KEY`를 읽는다.

```dotenv
OPENWEATHER_API_KEY=발급받은_키
```

`.env`는 Git에서 제외되며 실제 key를 Notebook 출력, AI Chat, MCP 등록 JSON에 작성하지 않는다. Server가 `.env`를 찾도록 PyCharm의 Working directory와 Terminal 위치를 `09_mcp`로 맞춘다. `mcpServers` JSON은 AI Assistant가 Weather MCP Server에 연결할 URL만 담고, OpenWeather 인증 정보는 담지 않는다.

PyCharm에서 `servers/weather_server.py`를 직접 실행하거나 `09_mcp` Terminal에서 다음 명령을 실행한다.

```bash
python servers/weather_server.py
```

Server가 `127.0.0.1:8000/mcp`에서 요청을 기다리면 AI Assistant에 다음 HTTP 설정을 추가한다.

```json
{
  "mcpServers": {
    "lessonWeather": {
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

`Apply`와 `Reconnect` 뒤 AI Chat에서 `lessonWeather의 current_weather Tool로 Seoul의 현재 날씨를 조회하고 전달한 city와 결과를 보여 줘`라고 요청한다. Tool 실행 기록에서 이름 `current_weather`, argument `city`, 구조화된 날씨 결과를 확인한다. 환경 변수를 바꾼 경우에는 Server 프로세스를 재시작해야 한다.


## Python Client에서 Weather Tool 연결 함수 만들기

앞에서 사용한 `Client`를 Weather endpoint에도 그대로 적용한다. 아래 셀의 URL 아래에 Tool 목록을 반환하는 발견 함수와 도시 이름을 `city` argument로 바꿔 `current_weather`를 호출하는 비동기 함수를 작성한다. 외부 API key는 이 Client 코드에 나타나지 않는다.


In [ ]:
WEATHER_MCP_URL = 'http://127.0.0.1:8000/mcp'


## Weather Tool 발견과 현재 날씨 호출 확인하기

먼저 `list_tools()` 결과에서 `current_weather`와 필수 입력 `city`를 확인한 뒤 실제 Tool을 호출한다. 온도·습도·설명은 실행 시점의 외부 데이터이므로 고정값을 정답으로 두지 않는다. 다음 셀에서 MCP 실행 상태와 반환 payload의 `city`, `temperature_c`, `humidity_percent`, `description` key가 존재하는지 검증한다.


## 정리

이번 실습에서는 Python 함수가 `MCPServer`와 `@mcp.tool()`을 통해 재사용 가능한 Tool 계약이 되는 흐름을 확인했다. JetBrains AI Assistant Host와 Notebook Python SDK Client는 역할은 다르지만 같은 Tool 이름, schema와 endpoint를 사용한다. Weather 선택 확장에서는 같은 Client 패턴을 유지하면서 외부 API key와 공급자 호출을 Server 내부에 분리한다.

다음 질문에 답하면서 실행 주체와 성공 근거를 구분한다.

1. Server 프로세스는 누가 시작했는가?
2. AI Assistant와 Notebook Client는 어느 endpoint에 연결했는가?
3. Tool 발견과 실제 Tool 호출은 각각 어떤 결과로 확인할 수 있는가?
4. Weather Client가 OpenWeatherMap API key를 몰라도 되는 이유는 무엇인가?
